# Functional Programming in Python

## Learning goals
By the end of this lesson, you should be able to:
- Explain what functional programming is and why it is useful.
- Recognize a pure function and a side effect.
- Pass a function to another function.
- Read and write a short `lambda` expression.
- Use `map()` to transform items, `filter()` to select items, and `reduce()` to combine items.
- Choose a clear alternative when a comprehension or built-in function is simpler.



## 1. What is functional programming?

Functional programming is a way of organizing a program around evaluating and combining functions. Instead of repeatedly changing shared data, we aim to describe how input values are transformed into results.

A central idea is the pure function. A pure function:
- Produces the same result for the same input values.
- Does not rely on changing external state.
- Does not cause observable side effects.

A side effect is an observable action beyond returning a result. Examples include changing an existing list, modifying a global variable, writing to a file, or displaying a message.

Python is not a purely functional language. It supports functional techniques alongside ordinary loops, mutable objects, and object-oriented programming. You can combine these approaches in one program.

## 2. What is its purpose?

Functional programming helps us express a task as a sequence of small, understandable operations on data.

Its practical purposes include:
- Clear transformations: describe what each value should become.
- Predictable behavior: reason about a pure function through its inputs and result.
- Easier testing: check a returned result without first recreating unrelated external state.
- Reuse: pass the same small function to different operations.
- Fewer unintended changes: prefer returning results when shared mutation is unnecessary.

A common data-processing task has three parts:
1. Select the items we need.
2. Transform those items.
3. Combine the results.

For example: keep the even numbers, double them, and calculate their total.

The purpose is not to eliminate every loop or make every expression shorter. Functional code is not automatically faster or clearer. Choose the form that makes the task easiest to understand.

## 3. Functions can be used as values

Python functions are first-class objects. This means we can assign a function to a variable, pass it as an argument, or return it from another function.

These capabilities let one function customize the behavior of another.

Notice the distinction:
- `double` refers to the function object.
- `double(4)` calls the function and produces `8`.

In [2]:
another_name = double

print(another_name is double)
print(another_name(4))

True
8


### Pass a function to another function

`apply_to_value()` receives a function and a value. It calls the supplied function on that value and returns the result.

Pass `double`, not `double()`: the receiving function needs the function object so it can call it later.

In [3]:
def apply_to_value(function, value):
    return function(value)

print(apply_to_value(double, 5))

10


A function that accepts or returns another function is called a higher-order function. The tools introduced later use this idea: we provide a function that tells the tool what operation to perform.

## 4. What is a lambda function?

A lambda expression creates a small anonymous function: a function that does not need a separate name at its point of use.

The syntax is:

```python
lambda parameters: expression
```

| Part | Meaning |
|---|---|
| `lambda` | Creates a function |
| `parameters` | Names that receive the arguments |
| `:` | Separates the parameters from the expression |
| `expression` | Calculates the value to return |

A lambda returns the value of its expression automatically. Do not write `return` inside it.

A lambda can have more than one parameter, but its body is a single expression—not a block of statements.

### Compare `def` and `lambda`

The following examples perform the same calculation. The named function is useful when we want to reuse it or explain it with a meaningful name. The lambda is useful for a short operation supplied directly to another function.

In [4]:
def square(number):
    return number ** 2

print(square(4))
print(apply_to_value(lambda number: number ** 2, 4))

16
16


### Use lambda inside a call

`sorted()` accepts a function through its `key` argument. It uses the value returned by that function to determine sorting order.

Here, each word's length becomes its sorting key. The result is a new list ordered from shortest to longest.

In [5]:
animals = ["ferret", "vole", "dog", "gecko"]

print(sorted(animals, key=lambda word: len(word)))
print(animals)

['dog', 'vole', 'gecko', 'ferret']
['ferret', 'vole', 'dog', 'gecko']


### When should you use lambda?

Use a lambda when the calculation is short, local, and easy to read. Use `def` when a descriptive name, documentation, several statements, or repeated use would help.

In the previous example, `key=len` is even simpler than the lambda because `len()` already performs the required task.

Lambda functions are not restricted to numeric calculations. They can return strings or other objects. They also do not guarantee purity: an expression can still call an operation with side effects.

## 5. Use `map()` to transform items

### Purpose
Use `map()` when you want to apply the same function to each item in an iterable, such as a list.

### Syntax

```python
map(function, iterable)
```

With one iterable, the supplied function receives one item at a time. `map()` returns an iterator that supplies the transformed results as they are requested.

It does not immediately return a list. Use `list()` to collect its results for display or later reuse.

In [19]:
numbers = [1, 2, 3, 4, 5]

doubled_numbers = list(map(double, numbers))

print(doubled_numbers)
print(numbers)

[2, 4, 6, 8, 10]
[1, 2, 3, 4, 5]


### How the transformation works

| Input item | Function call | Result |
|---|---|---|
| `1` | `double(1)` | `2` |
| `2` | `double(2)` | `4` |
| `3` | `double(3)` | `6` |

The remaining items are processed in the same way. The original list is unchanged because `double()` returns a result without mutating the input.

Read `map(double, numbers)` as: “apply `double` to each number.” The transformation function must be able to handle the items it receives.

### Use map with a lambda

A short transformation can be written directly in the call. Here, each number becomes its square.

In [7]:
squared_numbers = list(map(lambda number: number ** 2, numbers))
print(squared_numbers)

[1, 4, 9, 16, 25]


### Iterators are consumed

An iterator remembers its progress. `next()` obtains one result. Converting the iterator to a list then consumes the remaining results.

Once it is exhausted, it does not restart. Make a new `map()` object if you need to run the transformation again.

In [8]:
results = map(double, [1, 2, 3])

print(next(results))
print(list(results))
print(list(results))

2
[4, 6]
[]


## 6. Use `filter()` to select items

### Purpose
Use `filter()` when you want to keep only the items that satisfy a condition.

### Syntax

```python
filter(function, iterable)
```

The function tests each item. `filter()` keeps the original items for which the function returns a truthy result and discards the others.

A function used for this kind of test is often called a predicate. It commonly returns `True` or `False`.

Like `map()`, `filter()` returns an iterator. Use `list()` to collect the selected items.

In [9]:
def is_even(number):
    return number % 2 == 0

even_numbers = list(filter(is_even, numbers))

print(even_numbers)
print(numbers)

[2, 4]
[1, 2, 3, 4, 5]


### How selection works

| Item | Test | Keep the item? |
|---|---|---|
| `1` | `is_even(1)` → `False` | No |
| `2` | `is_even(2)` → `True` | Yes |
| `3` | `is_even(3)` → `False` | No |
| `4` | `is_even(4)` → `True` | Yes |
| `5` | `is_even(5)` → `False` | No |

The result is `[2, 4]`, not a list of Boolean values. `filter()` decides which items remain; it does not replace each item with its test result.

### Use filter with a lambda

A short condition can be written directly in the call. This example keeps numbers greater than 3.

In [10]:
greater_than_three = list(filter(lambda number: number > 3, numbers))
print(greater_than_three)

[4, 5]


### Distinguish transformation from selection

Use the same condition with both tools to see the difference:
- `map()` returns the result of the condition for every item.
- `filter()` returns the original items that pass the condition.

In [11]:
print(list(map(is_even, numbers)))
print(list(filter(is_even, numbers)))

[False, True, False, True, False]
[2, 4]


## 7. Use `reduce()` to combine items

### Purpose
Use `reduce()` when you want to repeatedly combine items into one accumulated result.

Unlike `map()` and `filter()`, `reduce()` is not a built-in function. Import it from the standard-library `functools` module.

### Syntax

```python
reduce(function, iterable)
```

The combining function must accept two arguments: the result so far and the next item.

Without an initial value, `reduce()` first combines the first two items. It then combines that result with the next item, continuing until no items remain. It returns the final result directly, not an iterator.

In [12]:
from functools import reduce

def add(first, second):
    return first + second

total = reduce(add, numbers)
print(total)

15


### Follow the accumulation

For `[1, 2, 3, 4, 5]`, addition proceeds from left to right:

| Step | Calculation | Running result |
|---|---|---|
| 1 | `1 + 2` | `3` |
| 2 | `3 + 3` | `6` |
| 3 | `6 + 4` | `10` |
| 4 | `10 + 5` | `15` |

The previous result becomes the first argument of the next call. This repeated combination is what makes the operation a reduction.

### Use reduce with a lambda

For a short combining operation, a lambda can replace the named function. It needs two parameters because the reduction combines two values at a time.

In [13]:
print(reduce(lambda first, second: first + second, numbers))

15


### Supply an initial value

A third argument provides a starting value:

```python
reduce(function, iterable, initial_value)
```

The first combination then uses the initial value and the first item. If the iterable is empty, the initial value is returned.

Without an initial value, an empty iterable raises `TypeError`. Choose a starting value that fits the operation: zero is appropriate for addition, but not for every calculation.

In [14]:
print(reduce(add, numbers, 100))
print(reduce(add, [], 0))

try:
    reduce(add, [])
except TypeError:
    print("An empty input needs an initial value.")

115
0
An empty input needs an initial value.


### Prefer a simpler built-in when it states the purpose clearly

Addition is useful for understanding `reduce()`, but `sum()` is the clearer choice when the task is simply to add numbers.

Similarly, use `max()` to find the largest value rather than writing a custom reduction.

A reduction produces one result object, but that object does not have to be a number. The examples here use numbers to keep the accumulation easy to follow.

In [15]:
print(sum(numbers))
print(max(numbers))

15
5


## 8. Put the operations together

Our task is to keep the even numbers, double them, and calculate their total.

Separate the three questions:
1. Which items should stay? Use `filter()`.
2. What should each remaining item become? Use `map()`.
3. How should the results be combined? Use `sum()` for addition.

Intermediate lists are created here so that we can inspect each step in class. They are not required by `map()` or `filter()`.

In [16]:
selected = list(filter(is_even, numbers))
transformed = list(map(double, selected))
result = sum(transformed)

print("Selected:", selected)
print("Transformed:", transformed)
print("Total:", result)

Selected: [2, 4]
Transformed: [4, 8]
Total: 12


## 9. Compare with a comprehension

A list comprehension can also express transformation and selection. It creates a list immediately.

The expression before `for` transforms each selected item. The `if` condition decides which items are included.

Neither style is compulsory. Prefer the version that you and your readers can explain most easily.

In [17]:
transformed_with_comprehension = [
    number * 2 for number in numbers if number % 2 == 0
]

print(transformed_with_comprehension)
print(sum(transformed_with_comprehension))

[4, 8]
12


## 10. Summary

| Feature | Purpose | Result |
|---|---|---|
| `lambda` | Define a short function using one expression | A function object |
| `map()` | Transform each item | An iterator of function results |
| `filter()` | Keep items that pass a test | An iterator of selected original items |
| `reduce()` | Repeatedly combine items | One accumulated result |

### Key ideas
- Functional programming emphasizes calculating results with functions rather than changing shared state.
- Pure functions make behavior easier to understand through inputs and outputs.
- Functions can be passed as values because they are first-class objects in Python.
- `lambda` is useful for short expressions; `def` is often clearer for named or complex logic.
- `map()` and `filter()` return iterators, which are consumed during iteration.
- `reduce()` comes from `functools`.
- Comprehensions and built-in functions may be clearer alternatives.
- Functional tools do not guarantee purity, clarity, or better performance by themselves.

## Practice

Use `numbers = [1, 2, 3, 4, 5]`.

1. Use `map()` with a lambda to triple every number.
2. Use `filter()` with a lambda to keep the odd numbers.
3. Use `reduce()` to multiply all the numbers together. Choose a suitable initial value.
4. Write comprehensions for tasks 1 and 2.
5. Explain which task is a transformation, which is a selection, and which is a reduction.

Predict the outputs before running your code.

In [18]:
# Write your solution here.